<img src="../../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<style>
h1 {
  font-size: 2.35rem !important;
  line-height: 1.15 !important;
  margin: 0.7rem 0 1rem !important;
}
h2 {
  font-size: 1.9rem !important;
  line-height: 1.2 !important;
  margin: 0.6rem 0 1rem !important;
}
h3 {
  font-size: 1.35rem !important;
  line-height: 1.3 !important;
  margin: 1.9rem 0 0.75rem !important;
}
h4 {
  font-size: 1.08rem !important;
  line-height: 1.35 !important;
  margin: 1.35rem 0 0.65rem !important;
}
.alchemi-api-label,
span[aria-label="ALCHEMI Toolkit API"] {
  color: currentColor !important;
  font-size: 0.70rem !important;
  font-weight: 700;
  letter-spacing: 0.05em;
}
@supports (color: color-mix(in srgb, black, white)) {
  .alchemi-api-label,
  span[aria-label="ALCHEMI Toolkit API"] {
    color: color-mix(in srgb, currentColor 68%, #76B900 32%) !important;
  }
}
</style>

# ALCHEMI Core · Module 01: Toolkit fundamentals and data structures

**Goal:** Represent one molecule as `AtomicData`, combine molecules with different atom counts in one Toolkit `Batch`, and inspect how their atom rows and boundaries are packed.

**Estimated time:** 30 minutes  
**Next module:** [Module 02: Simulate and compose →](alchemi-core-02-models-and-simulation.ipynb)


<img src="assets/course-roadmap-module01-atomicdata.svg" alt="ALCHEMI Core roadmap with Create AtomicData highlighted in Module 01" style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">


Module 00 showed several different molecules relaxed together. How does Toolkit represent one molecule and pack molecules with different atom counts together? One molecule becomes `AtomicData`, and several molecules become one `Batch`. Inside that `Batch`, each molecule is a graph, each atom is a node, and each model neighbor pair is an edge.

In [ ]:
import torch
from ase.build import molecule
from helpers import module01 as helpers
from nvalchemi.data import AtomicData, Batch


In [ ]:
helpers.configure_tutorial()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
if device.type == "cuda":
    print("gpu:   ", torch.cuda.get_device_name(device))

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 1 · Represent one molecule as `AtomicData`

If you completed 00, you just evaluated very different systems in one `Batch`. We will use four-atom ammonia to inspect the structure information ASE provides and the fields Toolkit stores in `AtomicData`.

### Build a small ASE molecule

`molecule("NH3")` returns an ASE `Atoms` object with element symbols and Cartesian positions in Å. Its `Atoms.info` mapping carries the net charge into `AtomicData.from_atoms(...)`.

In [ ]:
ammonia = molecule("NH3")
ammonia.info["charge"] = 0

Inspect the ASE structure before conversion: formula, atom count, net charge, coordinate unit, and positions.

In [ ]:
print("molecule:      ", ammonia)
print("formula:       ", ammonia.get_chemical_formula())
print("atoms:         ", len(ammonia))
print("charge:        ", ammonia.info["charge"], "e")
print("position unit: ", "Å")
print("positions:\n", ammonia.positions)

### See ammonia, now in 3D

Give the molecule a spin before converting it to `AtomicData`.

In [ ]:
helpers.show_molecule(ammonia, height=320)

### Convert ASE atoms to Toolkit `AtomicData`

`AtomicData.from_atoms(...)` converts an ASE `Atoms` object, validates the mapped fields, and places its tensors on the requested device. After conversion, inspect `atomic_numbers`, the shape of `positions`, the system-level `charge`, and `device`.

If a project uses pymatgen, `AtomicData.from_structure(...)` accepts a `Structure` or `Molecule`.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `AtomicData.from_atoms(atoms, device=device)`
>
> **Input:** an ASE `Atoms` object and a target device  
> **Result:** a new validated `AtomicData` object whose tensors live on that device

In [ ]:
ammonia_data = AtomicData.from_atoms(ammonia, device=device)

Inspect the new `AtomicData`: `atomic_numbers` identifies its four atoms, `positions.shape` shows one coordinate row per atom, `charge` belongs to the molecule, and `device` confirms tensor placement.

In [ ]:
print("atomic numbers:", ammonia_data.atomic_numbers.tolist())
print("positions:     ", ammonia_data.positions.shape)
print("charge:        ", ammonia_data.charge.item())
print("device:        ", ammonia_data.device)


**🔗 Learn more in Toolkit docs**

- [ASE integration](https://nvidia.github.io/nvalchemi-toolkit/examples/basic/03_ase_integration.html) - Convert real ASE molecules and structures into Toolkit data.
- [AtomicData and Batch example](https://nvidia.github.io/nvalchemi-toolkit/examples/basic/01_data_structures.html) - Explore direct construction, validated fields, chemical hashes, and device movement.

### ⚡ Power User: direct tensor construction and validation

*Construct `AtomicData` when tensors already exist.*

Open this optional section when a reader or simulation already supplies tensors. Direct construction makes you responsible for each field's shape, dtype, level, units, and device.

#### Identify the atoms

`atomic_numbers` contains one integer per atom. Here the values describe nitrogen followed by three hydrogens.

In [ ]:
# One atomic number per atom: N, H, H, H.
atomic_numbers = torch.tensor(
    [7, 1, 1, 1], dtype=torch.int32, device=device
)
print("atomic numbers:", atomic_numbers.tolist())
print("dtype:         ", atomic_numbers.dtype)
print("device:        ", atomic_numbers.device)

#### Add Cartesian coordinates

`AtomicData` stores coordinate values; the data source and model define their unit convention. This ASE example uses Å. One `[x, y, z]` row per atom gives shape `(4, 3)` for ammonia.

In [ ]:
# Four atom rows, with x, y, and z coordinates in Å.
positions = torch.tensor(
    [
        [0.000000,  0.000000,  0.116489],
        [0.000000,  0.939731, -0.271808],
        [0.813831, -0.469865, -0.271808],
        [-0.813831, -0.469865, -0.271808],
    ],
    dtype=torch.float32,
    device=device,
)
print("positions shape:", positions.shape)
print("position unit: ", "Å")

#### Add a system-level charge

The net charge belongs to the whole molecule. Toolkit represents this system-level scalar with shape `(1, 1)`.

In [ ]:
# Match the coordinate tensor's dtype and device.
charge = positions.new_zeros((1, 1))
print("charge:", charge)

#### Construct `AtomicData`

Pass the three tensors to the direct constructor. This creates a new `AtomicData` object on the tensors' current device.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `AtomicData(atomic_numbers=..., positions=..., charge=...)`
>
> **Input:** atom-level tensors plus optional system-level fields  
> **Result:** one validated `AtomicData` object

In [ ]:
direct_ammonia_data = AtomicData(
    atomic_numbers=atomic_numbers,
    positions=positions,
    charge=charge,
)
print("type:  ", type(direct_ammonia_data).__name__)
print("device:", direct_ammonia_data.device)

#### Compare the two construction routes

`AtomicData.chemical_hash` summarizes the composition, coordinates, periodicity, and cell. The native `==` operation compares this hash, so it can check whether both construction routes describe the same chemical system.

In [ ]:
print("ASE chemical hash:   ", ammonia_data.chemical_hash)
print("direct chemical hash:", direct_ammonia_data.chemical_hash)
print("hashes match:        ", ammonia_data.chemical_hash == direct_ammonia_data.chemical_hash)

#### Create a realistic shape mistake

A buggy reader may return coordinates as three coordinate rows by four atoms. Transposing `positions` reproduces that `(3, 4)` layout.

In [ ]:
transposed_positions = positions.T
print("positions shape:            ", tuple(positions.shape))
print("transposed positions shape: ", tuple(transposed_positions.shape))

#### Let validation locate the problem

Constructing `AtomicData` with the transposed coordinates runs the normal validator. The error identifies the incompatible positions shape.

In [ ]:
try:
    AtomicData(
        atomic_numbers=atomic_numbers,
        positions=transposed_positions,
        charge=charge,
    )
except ValueError as error:
    print(error)

#### Repair the coordinates

Transpose the reader output back to one coordinate row per atom, then run the same constructor again.


In [ ]:
AtomicData(
    atomic_numbers=atomic_numbers,
    positions=transposed_positions.T,
    charge=charge,
)
print("validation:", "passed")

**What to check:** validation catches `(3, 4)`, and `transposed_positions.T` with shape `(4, 3)` passes validation.

## 2 · Pack molecules into a `Batch`

<img src="assets/course-roadmap-module01-batch.svg" alt="ALCHEMI Core roadmap with Build a Batch highlighted in Module 01" style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0.75rem 0 0.5rem;">

Ammonia, propyne, and phenol contain 4, 7, and 13 atoms. Load the lesson's ASE propyne and phenol structures with `helpers.load_example_molecule(...)`, inspect each formula and atom count, then convert the three molecules to `AtomicData` and combine them with `Batch.from_data_list(...)`.

Inside a `Batch`, Toolkit represents each molecule/system as a **graph**, each atom as a **node**, and each model neighbor pair as an **edge**.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `Batch.from_data_list(data_list, device=device)`
>
> **Input:** validated `AtomicData` objects and a target device  
> **Result:** one `Batch` with contiguous atom rows and graph boundaries

In [ ]:
propyne = helpers.load_example_molecule("Propyne")
phenol = helpers.load_example_molecule("Phenol")
labels = ("Ammonia", "Propyne", "Phenol")

molecules = [ammonia, propyne, phenol]
# AIMNet2 requires each system's total charge; these three complexes are neutral.
for structure in molecules:
    structure.info["charge"] = 0

for label, structure in zip(labels, molecules, strict=True):
    print(f"{label}: {structure.get_chemical_formula()}, {len(structure)} atoms")

In [ ]:
# Reuse ammonia_data and convert propyne and phenol to AtomicData.
molecule_data = [ammonia_data] + [
    AtomicData.from_atoms(structure, device=device) for structure in molecules[1:]
]

`molecule_data` now holds one `AtomicData` object for each molecule. `Batch.from_data_list(...)` packs them into one `Batch` and records their graph boundaries.

In [ ]:
example_batch = Batch.from_data_list(molecule_data, device=device)

Inspect the packed `Batch`: `num_graphs` counts molecules, `num_nodes` counts atom rows, and `device` shows where the packed tensors live.

In [ ]:
print("molecules:", example_batch.num_graphs)
print("atoms:    ", example_batch.num_nodes)
print("device:   ", example_batch.device)

### See where each molecule begins and ends

Ammonia, propyne, and phenol contain 4, 7, and 13 atoms. Toolkit stores their atom rows back-to-back in one `Batch`. The total is 24 rows. **Packed graph batching avoids padding, so memory and model work scale with the atoms actually present.**

<img src="assets/01-cell-38-packed-and-padded-rows-fe8778d36be99922.svg" alt="Element-owner chips show ammonia rows 0 to 3, propyne rows 4 to 10, and phenol rows 11 to 23. The padded comparison adds blank chips until each system has 13 slots. Toolkit stores 24 rows; the padded layout stores 39 slots." style="display:block;box-sizing:border-box;width:100%;max-width:1104px;height:auto;margin:0.8rem 0 0.45rem;">

`batch_ptr` stores the four boundaries `[0, 4, 11, 24]`. `batch_idx` identifies which molecule contains each atom row.

> **Bottom line**
>
> A `Batch` packs molecules with different atom counts into contiguous rows and keeps their graph boundaries explicit.

`Batch.append(...)` changes the receiving `Batch` in place. The next cell clones `example_batch` first, so the original three-molecule batch remains available for the rest of the lesson.

In [ ]:
extended_batch = example_batch.clone()
print("molecules before append:", extended_batch.num_graphs)
extended_batch.append(example_batch)  # Changes extended_batch in place.
print("molecules after append: ", extended_batch.num_graphs)

#### Try it: build two kinds of batch

> ✏️ **TRY IT**
>
> Use the requested `BATCH_SIZE = 3`.
>
> 1. Pick three different ASE molecule names. Start with `"H2O"`, `"NH3"`, and `"CH4"`. Other choices include `"CO2"`, `"N2"`, `"O2"`, and `"C2H6"`.
> 2. Build a **homogeneous batch** by converting the first ASE molecule to `AtomicData` and repeating it `BATCH_SIZE` times.
> 3. Build a **heterogeneous batch** by converting all three ASE molecules to `AtomicData`.
>
> Each molecule becomes one graph in its `Batch`. `chemical_hash` summarizes its atomic numbers, positions, cell, and periodicity. Identical structures share one hash, so the check can distinguish repetition from three different molecules.
>
> **Success:** both batches match the requested batch size of three. The homogeneous batch has one unique `chemical_hash`; the heterogeneous batch has three.

In [ ]:
BATCH_SIZE = 3
MOLECULE_NAMES = ["H2O", "NH3", "CH4"]  # Choose BATCH_SIZE names.

# Reminder: AtomicData.from_atoms(molecule(name), device=device)
# TODO: build a homogeneous Batch with BATCH_SIZE copies of one molecule.
homogeneous_batch = None

# TODO: build a heterogeneous Batch from the selected molecules.
heterogeneous_batch = None

Run the check after completing the scaffold. `Batch.to_data_list()` recovers each graph as `AtomicData`; the unique hashes show whether the batch repeats one molecule or contains three different molecules.

In [ ]:
print("requested batch size:       ", BATCH_SIZE)
if homogeneous_batch is None or heterogeneous_batch is None:
    print("status: complete the TODOs, then run this check again")
else:
    homogeneous_hashes = {
        molecule_data.chemical_hash
        for molecule_data in homogeneous_batch.to_data_list()
    }
    heterogeneous_hashes = {
        molecule_data.chemical_hash
        for molecule_data in heterogeneous_batch.to_data_list()
    }
    print("homogeneous molecules:      ", homogeneous_batch.num_graphs)
    print("homogeneous unique hashes:  ", len(homogeneous_hashes))
    print("heterogeneous molecules:    ", heterogeneous_batch.num_graphs)
    print("heterogeneous unique hashes:", len(heterogeneous_hashes))

> ✅ **SAMPLE SOLUTION**

<details>
<summary><strong>Open the sample solution</strong></summary>

```python
homogeneous_data = [
    AtomicData.from_atoms(molecule("H2O"), device=device)
    for _ in range(BATCH_SIZE)
]
heterogeneous_data = [
    AtomicData.from_atoms(molecule(name), device=device)
    for name in MOLECULE_NAMES
]

homogeneous_batch = Batch.from_data_list(homogeneous_data, device=device)
heterogeneous_batch = Batch.from_data_list(heterogeneous_data, device=device)
```

The first list contains `BATCH_SIZE` water structures. The second converts each selected ASE molecule. Packing the lists produces a homogeneous batch with one unique chemical hash and a heterogeneous batch with three.

</details>

**🔗 Learn more in Toolkit docs**

- [AtomicData and Batch example](https://nvidia.github.io/nvalchemi-toolkit/examples/basic/01_data_structures.html) - Build, inspect, and select molecules in Toolkit batches.
- [ASE integration example](https://nvidia.github.io/nvalchemi-toolkit/examples/basic/03_ase_integration.html) - Convert ASE structures to `AtomicData` and back.

### ⚡ Power User: Batch fields, indexing, and slicing

*Inspect exact molecule and atom ownership inside the `Batch`.*

Open this optional section when an analysis needs individual molecules or exact atom-row ownership. In the `Batch` API, these are graph and node operations. Work through one property at a time, then use integer and slice indexing to recover molecules.

#### Count the atoms in each molecule

`num_nodes_per_graph` reports one atom count for each graph (molecule), in `Batch` order. For this example, the three values belong to ammonia, propyne, and phenol.

In [ ]:
# One atom count per molecule (graph), in Batch order.
print("atoms per molecule:", example_batch.num_nodes_per_graph.tolist())

#### Locate molecule boundaries

`batch_ptr` contains cumulative atom-row offsets between graphs (molecules). Read adjacent values as half-open spans: ammonia uses rows `0:4`, propyne uses `4:11`, and phenol uses `11:24`.

In [ ]:
# Cumulative start and stop offsets for atom rows.
print("atom boundaries:", example_batch.batch_ptr.tolist())

#### Map atoms to molecules

`batch_idx` contains one batch-local molecule index for each atom row. Each molecule is one graph and each atom row is one node, so a value of 0, 1, or 2 shows which molecule contains that atom.

In [ ]:
# One batch-local molecule index for each atom row.
print("molecule index per atom:", example_batch.batch_idx.tolist())

#### Visit every molecule

Use `batch[graph_index]` to recover one molecule as `AtomicData` since `batch` is **NOT** directly iterable like list. Loop over `range(batch.num_graphs)` to visit every molecule, or use a slice to create a smaller `Batch`.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `batch[index]`
>
> **Input:** one graph index or slice  
> **Result:** one `AtomicData` object or a smaller `Batch`

In [ ]:
# Visit each molecule by its graph index.
for graph_index in range(example_batch.num_graphs):
    molecule_data = example_batch[graph_index]
    print("molecule:   ", labels[graph_index])
    print("graph index:", graph_index)
    print("atoms:      ", molecule_data.num_nodes)

#### Select one molecule

Graph 2 represents phenol. Integer graph indexing returns this 13-atom molecule as one `AtomicData` object.

In [ ]:
# Integer graph indexing returns one molecule as AtomicData.
phenol_data = example_batch[2]
print("type: ", type(phenol_data).__name__)
print("atoms:", phenol_data.num_nodes)

#### Slice several molecules

A slice of graph indices returns a smaller `Batch`. The slice `1:3` keeps propyne and phenol in their original order.

In [ ]:
# A slice returns a smaller Batch.
last_two = example_batch[1:3]
print("type:     ", type(last_two).__name__)
print("molecules:", last_two.num_graphs)

**What to check:** the atom counts are `[4, 7, 13]`; the boundaries are `[0, 4, 11, 24]`; graph 2 represents 13-atom phenol; slicing graph indices 1:3 returns a `Batch` containing two molecules.

**🔗 Learn more in Toolkit docs**

- [AtomicData and Batch example](https://nvidia.github.io/nvalchemi-toolkit/examples/basic/01_data_structures.html) - Build batches, inspect molecule ownership, select molecules, slice batches, and append data.
- [Basic examples gallery](https://nvidia.github.io/nvalchemi-toolkit/examples/basic/index.html) - Continue with Toolkit data objects in complete molecular workflows.

### ⚡ Power User: save, load, and prefetch with Zarr

<img src="assets/course-roadmap-module01-prefetch.svg" alt="ALCHEMI Core roadmap with Load and prefetch highlighted in Module 01" style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0.75rem 0 0.5rem;">

*Keep larger collections on disk and prepare model-ready batches ahead of the GPU.*

Open this optional section when a larger workflow needs persistent records and background loading.

#### Why Zarr?

A simulation campaign can produce more structures than fit comfortably on a GPU - modern datasets span millions of structures and we want to use VRAM for simulations, not storage. Zarr stores arrays in chunks, so Toolkit can request selected records without loading the full collection. Toolkit also records field ownership and graph boundaries so every variable-size structure can be rebuilt as `AtomicData`.

In [ ]:
import shutil
import tempfile
from pathlib import Path

from nvalchemi.data import (
    AtomicDataZarrReader, AtomicDataZarrWriter,
    DataLoader, Dataset,
)

#### Create a temporary store

Use a temporary directory for this exercise. `AtomicDataZarrWriter` will create the Zarr store inside it.

In [ ]:
STORE = Path(tempfile.mkdtemp(prefix="alchemi-core-zarr-")) / "molecules.zarr"
writer = AtomicDataZarrWriter(STORE)

#### Save the three molecules

`write(...)` accepts the existing three-molecule `Batch` and stores each molecule as one record. Inside the `Batch`, each molecule is represented as a graph.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `writer.write(records)`
>
> **Input:** one `AtomicData`, a list of records, or a `Batch`  
> **Result:** a Zarr store with fields and per-record boundaries

In [ ]:
writer.write(example_batch)
print("stored molecules:", example_batch.num_graphs)

#### Open the saved collection

`AtomicDataZarrReader` opens the store and exposes its records without loading the full collection.

In [ ]:
reader = AtomicDataZarrReader(STORE)
print("available records:", len(reader))

#### Load records in a chosen order

`read_many(...)` returns raw CPU tensors. Request phenol first and ammonia second, then inspect their atom counts.

In [ ]:
requested = [2, 0]
loaded_records = reader.read_many(requested)
atom_counts = [len(fields["atomic_numbers"]) for fields, _ in loaded_records]
print("requested atom counts:", atom_counts)

#### Inspect stored field ownership

The reader records whether each saved field belongs to an atom, a model neighbor pair, or the whole molecule. Toolkit stores these as atom-, edge-, and system-level fields. Inside a `Batch`, atoms are nodes and model neighbor pairs are edges.

In [ ]:
print("field levels:", dict(sorted(reader.field_levels.items())))

#### Prepare a larger collection for prefetching

`Batch.to_data_list()` recovers the three molecules in `example_batch` before list repetition creates 3,000 records. Close the current reader, replace the small temporary store with the repeated records, and reopen it.

At the baseline settings below, the loader serves the collection in six model batches and three fused reads. The exercise then changes the fused read window.

In [ ]:
DATASET_REPEATS = 1000
expanded_records = example_batch.to_data_list() * DATASET_REPEATS

reader.close()
shutil.rmtree(STORE)
writer = AtomicDataZarrWriter(STORE)
writer.write(expanded_records)
reader = AtomicDataZarrReader(STORE)
print("stored molecules:", len(reader))

#### Why prefetching matters

For this repeated small-molecule dataset, each model `Batch` contains up to 512 molecules, or about 4,100 atom rows. With `prefetch_factor=2`, Toolkit fuses two emitted batches into one backend read:

`512 molecules per model batch × 2 batches per read = 1,024 molecules per fused read`

The loader reads up to 1,024 records together, then yields 512-molecule `Batch` objects to the model.

> **Why it matters**
>
> Fewer, larger reads reduce per-read overhead. Background prefetch starts the next fused read before yielding the current batches. With CUDA streams enabled, device transfers can also overlap with work on the current batch.

<img src="assets/01-cell-73-prefetch-window-v3-963036410d156b3e.svg" alt="Without prefetch, load and run chips alternate and one Batch is on the device. With prefetch factor 2, Batches 2 and 3 are prepared while earlier model work runs; device chips show Batch 1 current with Batches 2 and 3 ready." style="display:block;box-sizing:border-box;width:100%;max-width:1104px;height:auto;margin:0.8rem 0 0.45rem;">

#### Configure batching and prefetching

`Dataset` turns reader records into `AtomicData` on the selected device.

`num_workers=1` uses one background reader for this compact example. `shuffle=False` preserves stored order, while `use_streams=device.type == "cuda"` enables CUDA stream overlap only when CUDA is available.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `DataLoader(dataset, batch_size=..., prefetch_factor=..., use_streams=...)`
>
> **Input:** a Toolkit `Dataset` and loading settings  
> **Result:** an iterator that yields model-ready `Batch` objects; `prefetch_factor` controls how many emitted batches Toolkit fuses into each backend read

In [ ]:
MODEL_BATCH_SIZE = 512
PREFETCH_FACTOR = 2

dataset = Dataset(reader, device=device, num_workers=1)
loader = DataLoader(
    dataset, batch_size=MODEL_BATCH_SIZE, shuffle=False,
    prefetch_factor=PREFETCH_FACTOR,
    use_streams=device.type == "cuda",
)

#### Inspect the prefetch effect

Requesting the first item starts loading. The next cell reports the number of model batches, the molecules and atoms in the first batch, the prefetch factor, and the maximum fused read window.

`effective_read_window = batch_size × max(prefetch_factor, 1)` is the maximum number of molecules in one fused backend read.

In [ ]:
first_loaded_batch = next(iter(loader))
print("model batches:             ", len(loader))
print("molecules per model batch: ", first_loaded_batch.num_graphs)
print("atoms per model batch:     ", first_loaded_batch.num_nodes)
print("prefetch factor:           ", loader.prefetch_factor)
print("molecules per fused read:  ", loader.effective_read_window)

#### Try it: group four model batches per read

> ✏️ **TRY IT**
>
> Change `PREFETCH_FACTOR` so each fused backend read groups four total model batches. `MODEL_BATCH_SIZE` remains 512 molecules.
>
> Run the next cell and inspect the model batch size, prefetch factor, and fused read size.
>
> **Success:** the first model `Batch` contains 512 molecules, the loader reports `prefetch_factor=4`, and `effective_read_window` reports a maximum of 2,048 molecules.

In [ ]:
PREFETCH_FACTOR = 2  # TODO: group four model batches in each fused read.

wider_loader = DataLoader(
    dataset,
    batch_size=MODEL_BATCH_SIZE,
    shuffle=False,
    prefetch_factor=PREFETCH_FACTOR,
    use_streams=device.type == "cuda",
)
first_wider_batch = next(iter(wider_loader))

print("molecules per model batch: ", first_wider_batch.num_graphs)
print("prefetch factor:           ", wider_loader.prefetch_factor)
print("molecules per fused read:  ", wider_loader.effective_read_window)

> ✅ **SAMPLE SOLUTION**

<details>
<summary><strong>Open the sample solution</strong></summary>

```python
PREFETCH_FACTOR = 4
```

`prefetch_factor=4` groups four total model batches into each fused backend read. Each batch contains up to 512 molecules, so `effective_read_window` reports `512 × 4 = 2,048`. The loader yields those model `Batch` objects one at a time.

</details>

#### Clean up the temporary store

Close the Dataset workers before removing the temporary files.

In [ ]:
dataset.close()
shutil.rmtree(STORE.parent)
print("cleanup:", "temporary store removed")

**What to check:** the requested order produces atom counts `[13, 4]`; 1,000 repeats store 3,000 molecules; `MODEL_BATCH_SIZE = 512` produces six model batches; `prefetch_factor=2` gives a maximum fused read window of 1,024 molecules; and the Try It value `prefetch_factor=4` extends that maximum to 2,048 molecules.

**Return to the required path:** continue at **Module 1 recap**.

**🔗 Learn more in Toolkit docs**

- [Data loading guide](https://nvidia.github.io/nvalchemi-toolkit/userguide/datapipes.html) - Follow the complete `Reader → Dataset → DataLoader` path and understand background prefetching.
- [Zarr I/O example](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/02_trajectory_zarr_io.html) - Write molecular records to Zarr, reopen them, and replay stored data.
- [Large-dataset batching example](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/04_inflight_batching.html) - See how persistent data and live batches support a larger simulation campaign.

## Module 01 recap

You represented one molecule as `AtomicData`, combined three variable-size molecules in one `Batch`, and inspected their packed rows and graph boundaries.

The optional Zarr section saves the same records, loads them in a chosen order, and prefetches model-ready batches.

Module 02 uses the same Toolkit objects to simulate and compose molecular workflows.

**Next module:** [Module 02: Simulate and compose →](alchemi-core-02-models-and-simulation.ipynb)